# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing a dataset structured by Croissant using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their unique `@id` fields, consistent with best practices for interoperable dataset packages.

### Dataset Source
The dataset is defined in Croissant and hosted at:
[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

We'll use `mlcroissant` to load, overview, and explore FAIR^2 patient data: clinical tabulations, hormone levels, imaging, and genotype records.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()

print(f"\nDataset Name: {metadata['name']}\nDescription: {metadata['description']}\nPublished: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.
We'll inspect the record sets defined in this dataset.

In [ ]:
import json

# Reload the Croissant metadata as JSON for inspection
croissant_metadata = dataset.metadata.to_json()

# Print all available record sets by @id
record_sets = croissant_metadata.get('recordSet', [])
print("Available Record Sets (by @id):")
for record_set in record_sets:
    if isinstance(record_set, dict):
        print(f"  - {record_set['@id']} | name: {record_set.get('name', '<none>')}")
    else:
        print(f"  - {record_set}")

# Preview fields (columns) for each record set
for record_set in record_sets:
    rs_id = record_set['@id'] if isinstance(record_set, dict) else record_set
    print(f"\nRecord Set: {rs_id}")
    # Find fields in this record set
    fields = []
    if isinstance(record_set, dict):
        # Croissant stores fields under 'field' or 'column'
        fields = record_set.get('field', []) + record_set.get('column', [])
    if fields:
        for field in fields:
            fid = field['@id'] if isinstance(field, dict) else field
            fname = field.get('name', '<none>') if isinstance(field, dict) else '<none>'
            print(f"    * Field: {fname} (@id: {fid})")
    else:
        # As fallback, try listing top-level 'field' (Croissant schemas may differ)
        all_fields = croissant_metadata.get('field', []) + croissant_metadata.get('column', [])
        for field in all_fields:
            fid = field['@id'] if isinstance(field, dict) else field
            fname = field.get('name', '<none>') if isinstance(field, dict) else '<none>'
            print(f"    * Field: {fname} (@id: {fid})")

# Example usage: show a snapshot of one table
if len(record_sets) > 0:
    example_record_set_id = record_sets[0]['@id'] if isinstance(record_sets[0], dict) else record_sets[0]
    print(f"\nFirst record set: {example_record_set_id}\nSample record:")
    for x in dataset.records(record_set=example_record_set_id):
        print(x)
        break

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis. All references use `@id` for record sets and fields.
Below, we collect all available record sets and extract their records.

In [ ]:
# Prepare list of record set @id values
record_set_ids = [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets]
dataframes = {}

# Extract each record set to DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"\nLoaded record set '{record_set_id}' with shape: {dataframes[record_set_id].shape}")

    # Show available columns
    print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    # Preview first few records
    print(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA on one key record set. For example, we can filter, normalize, and group hormone measurements, or examine clinical features by patient.
All manipulation references columns by their `@id` where possible.

We'll select the first record set with numeric columns. We'll look for fields containing 'age', 'height', or hormone values, process those.

In [ ]:
# Choose a record set and a numeric field
selected_record_set_id = None
numeric_field_id = None

# Auto-choose (user can replace these)
for rs_id, df in dataframes.items():
    for col in df.columns:
        if 'age' in col.lower() or 'height' in col.lower() or 'hormone' in col.lower() or 'level' in col.lower():
            selected_record_set_id = rs_id
            numeric_field_id = col
            break
    if selected_record_set_id:
        break

print(f"Selected record set: {selected_record_set_id}")
print(f"Sample numeric field: {numeric_field_id}")

df = dataframes[selected_record_set_id]

# Filter records with the numeric field above a threshold
threshold = 10
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping (e.g., by 'patient_id', 'phenotype', etc.)
    group_field_id = None
    for col in df.columns:
        if 'patient' in col.lower() or 'phenotype' in col.lower():
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions or inter-field relationships. Let's plot the distribution of the selected numeric field and a grouped bar chart (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Plot distribution of numeric_field_id
if numeric_field_id and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8, 3))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=8)
    plt.title(f"Distribution of '{numeric_field_id}' in '{selected_record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # Visualize grouping if group_field_id computed
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrates loading and exploratory analysis of patient genotype-phenotype, hormone, and clinical features from the FAIR^2 dataset using the Croissant schema and `mlcroissant`.

- The dataset consists of three female patients with NC-21OHD, with rich fields across clinical, hormone, imaging, and genotype tables.
- All access and operations referenced record sets and fields by their `@id`, providing semantically robust data handling.
- Simple filtering and normalization operations highlight the structure and use cases for rare disease research.
- Visualizations reveal distributions and groupings in numeric measures.

Note: Due to small sample size, results are illustrative. For more advanced analysis, further clinical data and patient cohorts would be required.